In [2]:
# ============================================
# CELL 1: IMPORT & LOAD DATA
# ============================================
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

# Path
DATA_PATH = Path(r'D:\CAPSTONE\data\raw')
PROCESSED_PATH = Path(r'D:\CAPSTONE\data\processed')

# Load semua tabel
orders = pd.read_csv(DATA_PATH / 'olist_orders_dataset.csv')
order_items = pd.read_csv(DATA_PATH / 'olist_order_items_dataset.csv')
products = pd.read_csv(DATA_PATH / 'olist_products_dataset.csv')
customers = pd.read_csv(DATA_PATH / 'olist_customers_dataset.csv')
reviews = pd.read_csv(DATA_PATH / 'olist_order_reviews_dataset.csv')
category_translation = pd.read_csv(DATA_PATH / 'product_category_name_translation.csv')

print("✅ Semua data berhasil di-load!")
print(f"   orders      : {orders.shape}")
print(f"   order_items : {order_items.shape}")
print(f"   products    : {products.shape}")
print(f"   customers   : {customers.shape}")
print(f"   reviews     : {reviews.shape}")

✅ Semua data berhasil di-load!
   orders      : (99441, 8)
   order_items : (112650, 7)
   products    : (32951, 9)
   customers   : (99441, 5)
   reviews     : (99224, 7)


In [3]:
# ============================================
# CELL 2: CLEANING TABEL ORDERS
# ============================================
print("🧹 CLEANING: orders")
print(f"   Sebelum: {orders.shape}")

# --- Step 1: Konversi kolom tanggal ke datetime ---
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print("   ✅ Kolom tanggal dikonversi ke datetime")

# --- Step 2: Filter hanya order dengan status 'delivered' ---
# Kenapa? Karena untuk analisis BI & AI, kita butuh order yang
# benar-benar selesai (ada data delivery, ada review)
orders_clean = orders[orders['order_status'] == 'delivered'].copy()
print(f"   ✅ Filter status 'delivered': {len(orders_clean):,} rows")

# --- Step 3: Hapus baris yang delivered_date masih NULL ---
# (order delivered tapi tidak ada tanggal sampai = data tidak valid)
orders_clean = orders_clean.dropna(subset=['order_delivered_customer_date'])
print(f"   ✅ Hapus delivered_date NULL: {len(orders_clean):,} rows")

# --- Step 4: Verifikasi tidak ada duplikat order_id ---
dup = orders_clean.duplicated(subset=['order_id']).sum()
print(f"   ✅ Duplikat order_id: {dup} (harusnya 0)")

print(f"\n   Sesudah : {orders_clean.shape}")
print(f"   Rows dihapus: {orders.shape[0] - orders_clean.shape[0]:,}")

🧹 CLEANING: orders
   Sebelum: (99441, 8)
   ✅ Kolom tanggal dikonversi ke datetime
   ✅ Filter status 'delivered': 96,478 rows
   ✅ Hapus delivered_date NULL: 96,470 rows
   ✅ Duplikat order_id: 0 (harusnya 0)

   Sesudah : (96470, 8)
   Rows dihapus: 2,971


In [4]:
# ============================================
# CELL 3: CLEANING TABEL PRODUCTS
# ============================================
print("🧹 CLEANING: products")
print(f"   Sebelum: {products.shape}")

# --- Step 1: Lihat kolom mana yang missing ---
print("\n   Missing values per kolom:")
for col in products.columns:
    n = products[col].isnull().sum()
    if n > 0:
        print(f"   ⚠️  {col}: {n:,} missing ({n/len(products)*100:.1f}%)")

# --- Step 2: Isi missing category dengan 'unknown' ---
products_clean = products.copy()
products_clean['product_category_name'] = products_clean[
    'product_category_name'
].fillna('unknown')

print("\n   ✅ Missing category diisi dengan 'unknown'")

# --- Step 3: Translate kategori Portugis → Inggris ---
# Buat dictionary dari file translation
translation_dict = dict(
    zip(
        category_translation['product_category_name'],
        category_translation['product_category_name_english']
    )
)
# Tambahkan 'unknown' manual
translation_dict['unknown'] = 'unknown'

# Apply translation
products_clean['product_category_english'] = products_clean[
    'product_category_name'
].map(translation_dict)

# Cek kategori yang tidak tertranslate
not_translated = products_clean['product_category_english'].isnull().sum()
if not_translated > 0:
    print(f"   ⚠️  {not_translated} kategori tidak tertranslate → isi 'other'")
    products_clean['product_category_english'] = products_clean[
        'product_category_english'
    ].fillna('other')
else:
    print("   ✅ Semua kategori berhasil ditranslate ke Inggris")

# --- Step 4: Lihat hasil translation ---
print(f"\n   📋 Contoh hasil translation:")
sample = products_clean[['product_category_name', 'product_category_english']].drop_duplicates().head(8)
print(sample.to_string(index=False))

print(f"\n   Sesudah: {products_clean.shape}")

🧹 CLEANING: products
   Sebelum: (32951, 9)

   Missing values per kolom:
   ⚠️  product_category_name: 610 missing (1.9%)
   ⚠️  product_name_lenght: 610 missing (1.9%)
   ⚠️  product_description_lenght: 610 missing (1.9%)
   ⚠️  product_photos_qty: 610 missing (1.9%)
   ⚠️  product_weight_g: 2 missing (0.0%)
   ⚠️  product_length_cm: 2 missing (0.0%)
   ⚠️  product_height_cm: 2 missing (0.0%)
   ⚠️  product_width_cm: 2 missing (0.0%)

   ✅ Missing category diisi dengan 'unknown'
   ⚠️  13 kategori tidak tertranslate → isi 'other'

   📋 Contoh hasil translation:
product_category_name product_category_english
           perfumaria                perfumery
                artes                      art
        esporte_lazer           sports_leisure
                bebes                     baby
utilidades_domesticas               housewares
instrumentos_musicais      musical_instruments
           cool_stuff               cool_stuff
     moveis_decoracao          furniture_decor

   Ses

In [5]:
# ============================================
# CELL 4: CLEANING TABEL REVIEWS
# ============================================
print("🧹 CLEANING: reviews")
print(f"   Sebelum: {reviews.shape}")

reviews_clean = reviews.copy()

# --- Step 1: Konversi tanggal ---
reviews_clean['review_creation_date'] = pd.to_datetime(
    reviews_clean['review_creation_date'], errors='coerce'
)
print("   ✅ Kolom tanggal dikonversi")

# --- Step 2: Hapus duplikat review ---
# Satu order harusnya hanya punya satu review
dup_before = reviews_clean.duplicated(subset=['order_id']).sum()
reviews_clean = reviews_clean.drop_duplicates(subset=['order_id'], keep='last')
print(f"   ✅ Duplikat order_id dihapus: {dup_before:,} rows")

# --- Step 3: Buat kolom sentiment label dari review_score ---
# Ini BUKAN AI — ini hanya label untuk training data Sentiment NLP
def label_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

reviews_clean['sentiment_label'] = reviews_clean['review_score'].apply(label_sentiment)
print("   ✅ Kolom sentiment_label dibuat dari review_score")

# --- Step 4: Pisahkan reviews yang punya teks ---
# Ini yang akan dipakai untuk training Sentiment Analysis AI
reviews_with_text = reviews_clean[
    reviews_clean['review_comment_message'].notna()
].copy()

print(f"\n   📊 Distribusi review score:")
score_count = reviews_clean['review_score'].value_counts().sort_index()
for score, count in score_count.items():
    pct = count/len(reviews_clean)*100
    print(f"      Score {score}: {count:>6,} ({pct:.1f}%)")

print(f"\n   📊 Distribusi sentiment label:")
sentiment_count = reviews_clean['sentiment_label'].value_counts()
for label, count in sentiment_count.items():
    pct = count/len(reviews_clean)*100
    print(f"      {label:<10}: {count:>6,} ({pct:.1f}%)")

print(f"\n   📝 Reviews dengan teks: {len(reviews_with_text):,}")
print(f"      (Ini data untuk Sentiment Analysis AI)")

print(f"\n   Sesudah: {reviews_clean.shape}")

🧹 CLEANING: reviews
   Sebelum: (99224, 7)
   ✅ Kolom tanggal dikonversi
   ✅ Duplikat order_id dihapus: 551 rows
   ✅ Kolom sentiment_label dibuat dari review_score

   📊 Distribusi review score:
      Score 1: 11,356 (11.5%)
      Score 2:  3,128 (3.2%)
      Score 3:  8,131 (8.2%)
      Score 4: 19,046 (19.3%)
      Score 5: 57,012 (57.8%)

   📊 Distribusi sentiment label:
      positive  : 76,058 (77.1%)
      negative  : 14,484 (14.7%)
      neutral   :  8,131 (8.2%)

   📝 Reviews dengan teks: 40,783
      (Ini data untuk Sentiment Analysis AI)

   Sesudah: (98673, 8)


In [6]:
# ============================================
# CELL 5: VERIFIKASI HASIL CLEANING
# ============================================
print("="*60)
print("✅ VERIFIKASI HASIL DATA CLEANING")
print("="*60)

cleaned_data = {
    'orders_clean': orders_clean,
    'products_clean': products_clean,
    'customers': customers,        # sudah bersih
    'order_items': order_items,    # sudah bersih
    'reviews_clean': reviews_clean,
}

print(f"\n{'Nama':<20} {'Sebelum':>10} {'Sesudah':>10} {'Berkurang':>10}")
print("-"*55)

before = {
    'orders_clean': 99441,
    'products_clean': 32951,
    'customers': 99441,
    'order_items': 112650,
    'reviews_clean': 99224,
}

for name, df in cleaned_data.items():
    b = before[name]
    a = df.shape[0]
    diff = b - a
    print(f"{name:<20} {b:>10,} {a:>10,} {diff:>10,}")

print("\n✅ Semua missing values sudah ditangani")
print("✅ Semua duplikat sudah dihapus")
print("✅ Semua format tanggal sudah benar")
print("✅ Kategori produk sudah ditranslate ke Inggris")
print("✅ Sentiment label sudah dibuat untuk training AI")

✅ VERIFIKASI HASIL DATA CLEANING

Nama                    Sebelum    Sesudah  Berkurang
-------------------------------------------------------
orders_clean             99,441     96,470      2,971
products_clean           32,951     32,951          0
customers                99,441     99,441          0
order_items             112,650    112,650          0
reviews_clean            99,224     98,673        551

✅ Semua missing values sudah ditangani
✅ Semua duplikat sudah dihapus
✅ Semua format tanggal sudah benar
✅ Kategori produk sudah ditranslate ke Inggris
✅ Sentiment label sudah dibuat untuk training AI


In [7]:
# ============================================
# CELL 6: SIMPAN HASIL CLEANING
# ============================================
print("💾 Menyimpan hasil cleaning...\n")

# Simpan ke folder processed
orders_clean.to_csv(
    PROCESSED_PATH / 'orders_clean.csv', index=False
)
print(f"   ✅ orders_clean.csv saved ({len(orders_clean):,} rows)")

products_clean.to_csv(
    PROCESSED_PATH / 'products_clean.csv', index=False
)
print(f"   ✅ products_clean.csv saved ({len(products_clean):,} rows)")

reviews_clean.to_csv(
    PROCESSED_PATH / 'reviews_clean.csv', index=False
)
print(f"   ✅ reviews_clean.csv saved ({len(reviews_clean):,} rows)")

reviews_with_text.to_csv(
    PROCESSED_PATH / 'reviews_with_text.csv', index=False
)
print(f"   ✅ reviews_with_text.csv saved ({len(reviews_with_text):,} rows)")

# customers & order_items sudah bersih, simpan langsung
customers.to_csv(
    PROCESSED_PATH / 'customers_clean.csv', index=False
)
print(f"   ✅ customers_clean.csv saved ({len(customers):,} rows)")

order_items.to_csv(
    PROCESSED_PATH / 'order_items_clean.csv', index=False
)
print(f"   ✅ order_items_clean.csv saved ({len(order_items):,} rows)")

print("\n🎉 Notebook 02 selesai!")
print("➡️  Langkah berikutnya: 03_feature_engineering.ipynb")
print(f"\n📁 File tersimpan di: {PROCESSED_PATH}")

💾 Menyimpan hasil cleaning...

   ✅ orders_clean.csv saved (96,470 rows)
   ✅ products_clean.csv saved (32,951 rows)
   ✅ reviews_clean.csv saved (98,673 rows)
   ✅ reviews_with_text.csv saved (40,783 rows)
   ✅ customers_clean.csv saved (99,441 rows)
   ✅ order_items_clean.csv saved (112,650 rows)

🎉 Notebook 02 selesai!
➡️  Langkah berikutnya: 03_feature_engineering.ipynb

📁 File tersimpan di: D:\CAPSTONE\data\processed
